# The Sun in Time: Activity and Environment — Implementation
# 시간 속의 태양: 활동과 환경 — 구현

**Paper**: Güdel, M. (2007), "The Sun in Time: Activity and Environment", *Living Rev. Solar Phys.*, 4, 3

This notebook implements the key quantitative relations from Güdel's comprehensive review of solar activity evolution over 4.6 Gyr:
이 노트북은 Güdel의 46억 년 태양 활동 진화 종합 리뷰에서 핵심 정량적 관계들을 구현합니다:

1. **Rotational Spin-down (Skumanich Law)** — 자전 감속: P vs age, v_rot vs age
2. **Activity-Rotation-Age Relations** — 활동-자전-나이 관계: L_X vs P, L_X vs age
3. **Multi-wavelength Radiation Decay** — 다파장 복사 감쇠: X-ray/EUV/FUV/UV flux vs age
4. **Spectral Irradiance Evolution** — 분광 복사 진화: spectral softening over time
5. **Coronal Temperature Evolution** — 코로나 온도 진화: T_av vs L_X
6. **Faint Young Sun Paradox** — 젊고 어두운 태양 역설: bolometric luminosity vs age
7. **Planetary Atmospheric Escape** — 행성 대기 탈출: thermal escape parameter for Venus/Earth/Mars

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter

plt.rcParams.update({
    "figure.dpi": 120,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## Part 1: Rotational Spin-down (Skumanich Law)
## 파트 1: 자전 감속 (Skumanich 법칙)

Güdel Eq. 7 gives the rotation period as a function of age:
Güdel Eq. 7은 나이의 함수로 자전 주기를 제공합니다:

$$P = 0.21\,t_6^{0.57} \quad \text{[days]}$$

where $t_6$ is the age after ZAMS in Myr. Eq. 8 gives the angular velocity decay:
여기서 $t_6$는 ZAMS 이후 나이(Myr 단위)입니다. Eq. 8은 각속도 감쇠를 제공합니다:

$$\Omega \propto v_\text{rot} \propto t_6^{-0.6 \pm 0.1}$$

The original Skumanich (1972) law gave $\Omega \propto t^{-0.5}$; Güdel's refinement is slightly steeper at $-0.6$.
원래 Skumanich (1972) 법칙은 $\Omega \propto t^{-0.5}$; Güdel의 개선된 값은 $-0.6$으로 약간 더 가파릅니다.

We also overlay the "Sun in Time" sample stars from Table 3.
Table 3의 "Sun in Time" 샘플 별들을 함께 표시합니다.

In [ ]:
# === Rotational Spin-down: P vs age, v_rot vs age (Güdel Eqs. 7-8) ===

# "Sun in Time" sample stars (Table 3).
# Names, ages [Gyr], rotation periods [days].
sit_names = ["EK Dra", r"$\pi^1$ UMa", r"$\chi^1$ Ori",
             r"$\kappa^1$ Cet", r"$\beta$ Com", "Sun", r"$\beta$ Hyi"]
sit_ages_gyr = np.array([0.1, 0.3, 0.3, 0.65, 1.6, 4.6, 6.7])
sit_prot_d = np.array([2.68, 4.90, 5.24, 9.21, 12.4, 25.4, 28.0])

# Continuous age grid (Myr for Eq. 7).
age_myr = np.logspace(np.log10(50), np.log10(10000), 500)  # 50 Myr – 10 Gyr
age_gyr = age_myr / 1000.0

# Eq. 7: P = 0.21 * t_6^{0.57} [days], t_6 in Myr.
P_eq7 = 0.21 * age_myr ** 0.57

# Equatorial rotation velocity: v = 2 pi R_sun / P.
R_sun_km = 6.957e5  # km
v_rot = 2.0 * np.pi * R_sun_km / (P_eq7 * 86400.0)  # km/s
sit_vrot = 2.0 * np.pi * R_sun_km / (sit_prot_d * 86400.0)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: P_rot vs age ---
ax = axes[0]
ax.loglog(age_gyr, P_eq7, "b-", lw=2, label="Eq. 7: $P = 0.21\\,t_6^{0.57}$")
ax.scatter(sit_ages_gyr, sit_prot_d, c="red", s=80, zorder=5, edgecolors="k")
for name, x, y in zip(sit_names, sit_ages_gyr, sit_prot_d):
    ax.annotate(name, (x, y), textcoords="offset points",
                xytext=(6, 6), fontsize=8)
ax.set_xlabel("Age [Gyr] / 나이 [Gyr]")
ax.set_ylabel("Rotation Period [days] / 자전 주기 [일]")
ax.set_title("Rotation Period vs Age\n자전 주기 vs 나이 (Fig. 11 style)")
ax.legend(fontsize=9)
ax.set_xlim(0.05, 10)
ax.set_ylim(1, 50)

# --- Right: v_rot vs age ---
ax = axes[1]
ax.loglog(age_gyr, v_rot, "b-", lw=2,
          label=r"$v_\mathrm{rot} \propto t^{-0.57}$ (from Eq. 7)")
# Original Skumanich for comparison.
v_skum = v_rot[0] * (age_gyr / age_gyr[0]) ** (-0.5)
ax.loglog(age_gyr, v_skum, "g--", lw=1.5, alpha=0.7,
          label=r"Skumanich: $v \propto t^{-0.5}$")
ax.scatter(sit_ages_gyr, sit_vrot, c="red", s=80, zorder=5, edgecolors="k")
for name, x, y in zip(sit_names, sit_ages_gyr, sit_vrot):
    ax.annotate(name, (x, y), textcoords="offset points",
                xytext=(6, 6), fontsize=8)
ax.set_xlabel("Age [Gyr] / 나이 [Gyr]")
ax.set_ylabel(r"$v_\mathrm{rot}$ [km/s]")
ax.set_title("Equatorial Rotation Velocity vs Age\n적도 자전 속도 vs 나이")
ax.legend(fontsize=9)
ax.set_xlim(0.05, 10)

plt.tight_layout()
plt.show()

# Print summary table.
print("Sun in Time Sample — Rotation Summary")
print(f"{'Star':<15} {'Age [Gyr]':>10} {'P [d]':>8} {'v [km/s]':>10}")
print("-" * 48)
for n, a, p, v in zip(sit_names, sit_ages_gyr, sit_prot_d, sit_vrot):
    # Strip LaTeX for table display.
    clean = n.replace("$", "").replace(r"\pi^1", "pi1").replace(
        r"\chi^1", "chi1").replace(r"\kappa^1", "kap1").replace(
        r"\beta", "beta")
    print(f"{clean:<15} {a:>10.2f} {p:>8.2f} {v:>10.2f}")

## Part 2: Activity-Rotation-Age Relations
## 파트 2: 활동-자전-나이 관계

The core quantitative results of the paper — linking rotation to X-ray luminosity and both to age.
논문의 핵심 정량적 결과 — 자전을 X선 광도에, 그리고 둘 다 나이에 연결합니다.

**Eq. 13** (L_X vs rotation period, Fig. 14 style):
$$L_X = 10^{31.05 \pm 0.12}\,P^{-2.64 \pm 0.12} \quad \text{[erg s}^{-1}\text{]}$$

**Eq. 12** (L_X vs age):
$$L_X \approx (3 \pm 1) \times 10^{28}\,t_9^{-1.5 \pm 0.3} \quad \text{[erg s}^{-1}\text{]}$$

The saturation level is at $L_X/L_\text{bol} \approx 10^{-3}$, i.e., $L_X^\text{sat} \approx 3.8 \times 10^{30}$ erg/s for a solar-luminosity star.
포화 수준은 $L_X/L_\text{bol} \approx 10^{-3}$, 즉 태양 광도 별의 경우 $L_X^\text{sat} \approx 3.8 \times 10^{30}$ erg/s입니다.

In [ ]:
# === Activity-Rotation-Age Relations (Güdel Eqs. 12-13, Fig. 14) ===

L_bol_sun = 3.828e33  # erg/s (solar bolometric luminosity)
L_X_sat = 1e-3 * L_bol_sun  # Saturation level.

# --- Left panel: L_X vs P (Eq. 13, Fig. 14 style) ---
P_grid = np.logspace(np.log10(0.5), np.log10(50), 300)  # days
L_X_eq13 = 10**31.05 * P_grid**(-2.64)

# Clamp at saturation.
L_X_eq13_sat = np.minimum(L_X_eq13, L_X_sat)

# Sample stars: approximate L_X from their P_rot using Eq. 13.
sit_LX_from_P = 10**31.05 * sit_prot_d**(-2.64)
sit_LX_from_P = np.minimum(sit_LX_from_P, L_X_sat)

# --- Right panel: L_X vs age (Eq. 12) ---
age_gyr_grid = np.logspace(np.log10(0.05), np.log10(10), 300)
L_X_eq12 = 3e28 * age_gyr_grid**(-1.5)
L_X_eq12_sat = np.minimum(L_X_eq12, L_X_sat)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# --- Left: L_X vs P ---
ax = axes[0]
ax.loglog(P_grid, L_X_eq13, "b-", lw=2, alpha=0.4,
          label=r"Eq. 13: $L_X = 10^{31.05}\,P^{-2.64}$")
ax.loglog(P_grid, L_X_eq13_sat, "b-", lw=2.5,
          label="With saturation / 포화 포함")
ax.axhline(L_X_sat, color="gray", ls="--", lw=1,
           label=r"$L_X/L_\mathrm{bol}=10^{-3}$ saturation")
ax.scatter(sit_prot_d, sit_LX_from_P, c="red", s=80, zorder=5, edgecolors="k")
for name, x, y in zip(sit_names, sit_prot_d, sit_LX_from_P):
    ax.annotate(name, (x, y), textcoords="offset points",
                xytext=(5, 7), fontsize=8)
# Mark the present Sun explicitly.
L_X_sun = 1e27  # erg/s (approximate mean solar L_X)
ax.scatter([25.4], [L_X_sun], marker="*", s=200, c="gold",
           edgecolors="k", zorder=6, label=r"Sun (observed $L_X$)")
ax.set_xlabel("Rotation Period $P$ [days] / 자전 주기 [일]")
ax.set_ylabel(r"$L_X$ [erg s$^{-1}$]")
ax.set_title(r"$L_X$ vs Rotation Period (Fig. 14 style)" "\n"
             r"$L_X$ vs 자전 주기")
ax.legend(fontsize=8, loc="upper right")
ax.set_xlim(0.5, 50)
ax.set_ylim(1e26, 1e31)

# --- Right: L_X vs age ---
ax = axes[1]
ax.loglog(age_gyr_grid, L_X_eq12, "b-", lw=2, alpha=0.4,
          label=r"Eq. 12: $L_X = 3\times10^{28}\,t_9^{-1.5}$")
ax.loglog(age_gyr_grid, L_X_eq12_sat, "b-", lw=2.5,
          label="With saturation / 포화 포함")
ax.axhline(L_X_sat, color="gray", ls="--", lw=1)
ax.scatter(sit_ages_gyr, sit_LX_from_P, c="red", s=80, zorder=5,
           edgecolors="k")
for name, x, y in zip(sit_names, sit_ages_gyr, sit_LX_from_P):
    ax.annotate(name, (x, y), textcoords="offset points",
                xytext=(5, 7), fontsize=8)
ax.scatter([4.6], [L_X_sun], marker="*", s=200, c="gold",
           edgecolors="k", zorder=6, label=r"Sun (observed $L_X$)")
ax.set_xlabel("Age [Gyr] / 나이 [Gyr]")
ax.set_ylabel(r"$L_X$ [erg s$^{-1}$]")
ax.set_title(r"$L_X$ Decay with Age" "\n"
             "X선 광도의 나이에 따른 감쇠")
ax.legend(fontsize=8, loc="upper right")
ax.set_xlim(0.05, 10)
ax.set_ylim(1e26, 1e31)

plt.tight_layout()
plt.show()

# Print enhancement factors at key epochs.
print("\nX-ray Enhancement Factors (relative to present Sun)")
print("X선 향상 계수 (현재 태양 대비)")
print(f"{'Age [Gyr]':>10} {'L_X [erg/s]':>14} {'Enhancement':>12}")
print("-" * 40)
for t in [0.1, 0.3, 0.5, 1.0, 2.0, 3.5, 4.6]:
    lx = min(3e28 * t**(-1.5), L_X_sat)
    print(f"{t:>10.1f} {lx:>14.2e} {lx / L_X_sun:>12.0f}×")

## Part 3: Multi-wavelength Radiation Decay
## 파트 3: 다파장 복사 감쇠

Güdel's key finding: all wavelength bands decay as power laws with age (Eq. 17), but **shorter wavelengths decay faster** (Table 5, Fig. 19). The decay slope depends on the formation temperature of each band (Eq. 18):

Güdel의 핵심 발견: 모든 파장대는 나이에 따라 멱급수로 감쇠하지만 (Eq. 17), **단파장일수록 더 빠르게 감쇠합니다** (Table 5, Fig. 19). 감쇠 기울기는 각 파장대의 생성 온도에 의존합니다 (Eq. 18):

$$F = \alpha\,t_9^{\beta} \quad \text{(Eq. 17)}$$

$$-\beta = -0.46 + 0.32\,\log T_\text{max} \quad \text{(Eq. 18)}$$

In [ ]:
# === Multi-wavelength Radiation Decay (Eq. 17, Table 5, Fig. 19) ===

# Table 5 data: band name, wavelength range label, beta exponent,
# log T_max [K], color for plotting.
bands = [
    ("X-ray (1-20 A)",      -1.92, 7.0,  "purple"),
    ("Soft X-ray (20-100 A)", -1.39, 6.5,  "blue"),
    ("EUV (100-360 A)",     -1.20, 6.0,  "dodgerblue"),
    ("FUV (920-1180 A)",    -0.85, 5.0,  "green"),
    ("Ly-alpha (1216 A)",   -0.72, 4.4,  "limegreen"),
    ("UV lines (1200-1700 A)", -0.72, 4.5, "orange"),
    ("UV cont. (1700-3100 A)", -0.27, 3.7, "red"),
]

age_gyr = np.logspace(np.log10(0.07), np.log10(10), 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# --- Left: Flux enhancement vs age (Fig. 19 style) ---
ax = axes[0]
for name, beta, log_t, color in bands:
    # Normalize so that F(4.6 Gyr) = 1.
    enhancement = (age_gyr / 4.6) ** beta
    ax.loglog(age_gyr, enhancement, lw=2.2, color=color,
              label=f"{name} ($\\beta$={beta:.2f})")

ax.axhline(1, color="k", ls=":", lw=0.8)
ax.axvline(4.6, color="k", ls=":", lw=0.8, alpha=0.5)
ax.scatter([4.6], [1], marker="*", s=200, c="gold", edgecolors="k", zorder=6)
ax.annotate("Present Sun\n현재 태양", (4.6, 1), textcoords="offset points",
            xytext=(10, -20), fontsize=8)
ax.set_xlabel("Age [Gyr] / 나이 [Gyr]")
ax.set_ylabel("Flux / Present Solar Flux\n플럭스 / 현재 태양 플럭스")
ax.set_title("Multi-wavelength Flux Decay (Fig. 19 style)\n다파장 복사 감쇠")
ax.legend(fontsize=7, loc="upper right", ncol=1)
ax.set_xlim(0.07, 10)
ax.set_ylim(0.3, 3000)

# --- Right: Decay slope vs formation temperature (Eq. 18) ---
ax = axes[1]
log_T_arr = np.array([b[2] for b in bands])
neg_beta_arr = np.array([-b[1] for b in bands])
colors_arr = [b[3] for b in bands]

ax.scatter(log_T_arr, neg_beta_arr, c=colors_arr, s=100, edgecolors="k",
           zorder=5)
for (name, beta, log_t, color) in bands:
    ax.annotate(name.split("(")[0].strip(), (log_t, -beta),
                textcoords="offset points", xytext=(5, 5), fontsize=7)

# Eq. 18 fit line.
log_T_line = np.linspace(3.5, 7.5, 100)
neg_beta_line = -0.46 + 0.32 * log_T_line
ax.plot(log_T_line, neg_beta_line, "k--", lw=1.5,
        label=r"Eq. 18: $-\beta = -0.46 + 0.32\,\log T_\mathrm{max}$")

ax.set_xlabel(r"$\log\,T_\mathrm{max}$ [K] / 생성 온도")
ax.set_ylabel(r"$-\beta$ (decay slope / 감쇠 기울기)")
ax.set_title("Decay Slope vs Formation Temperature (Eq. 18)\n"
             "감쇠 기울기 vs 생성 온도")
ax.legend(fontsize=9)
ax.set_xlim(3.3, 7.5)
ax.set_ylim(-0.5, 2.5)

plt.tight_layout()
plt.show()

# Print Table 5 summary.
print("\nTable 5 Summary — Power-law Decay Parameters")
print(f"{'Band':<28} {'beta':>8} {'log T_max':>10} {'Enh. at 0.1 Gyr':>16}")
print("-" * 65)
for name, beta, log_t, _ in bands:
    enh = (0.1 / 4.6) ** beta
    print(f"{name:<28} {beta:>8.2f} {log_t:>10.1f} {enh:>16.0f}x")

## Part 4: Spectral Irradiance Evolution
## 파트 4: 분광 복사 진화

Reproducing the essence of **Figure 20**: showing how the solar spectral irradiance softens over time.
**Figure 20**의 본질 재현: 태양 분광 복사가 시간에 따라 어떻게 부드러워지는지 보여줍니다.

The young Sun's spectrum is "harder" — relatively much stronger at short (X-ray/EUV) wavelengths than at long (UV/visible) wavelengths. As the Sun ages, the short-wavelength excess fades fastest, and the spectrum "softens."

젊은 태양의 스펙트럼은 "더 단단" — 긴 파장(UV/가시광)보다 짧은 파장(X선/EUV)에서 상대적으로 훨씬 더 강합니다. 태양이 나이 들면서 짧은 파장의 초과분이 가장 빨리 사라지며, 스펙트럼이 "부드러워집니다."

In [ ]:
# === Spectral Irradiance Evolution (Fig. 20 style) ===

# Representative wavelength grid and associated beta exponents.
# We use Eq. 18 to assign a continuous beta(lambda) via T_max(lambda).
# Approximate mapping: shorter wavelength -> higher T_max -> steeper beta.
# We define discrete "spectral bins" with representative wavelengths.
spec_bins = [
    # (representative wavelength [A], band label, beta, present flux [arb])
    (5,    "Hard X-ray",     -1.92, 0.01),
    (20,   "Soft X-ray",    -1.70, 0.05),
    (50,   "XUV",           -1.39, 0.15),
    (150,  "EUV",           -1.20, 0.5),
    (400,  "EUV/FUV",       -1.00, 1.0),
    (700,  "FUV",           -0.85, 3.0),
    (1000, "FUV",           -0.80, 8.0),
    (1216, "Ly-alpha",      -0.72, 15.0),
    (1500, "UV lines",      -0.72, 20.0),
    (2000, "UV cont.",      -0.40, 50.0),
    (2500, "NUV",           -0.27, 100.0),
    (3000, "NUV",           -0.20, 200.0),
    (4000, "Visible",       -0.05, 500.0),
    (5000, "Visible",       -0.02, 800.0),
    (6000, "Visible",       -0.01, 700.0),
    (8000, "NIR",            0.00, 400.0),
]

wavelengths = np.array([s[0] for s in spec_bins])
betas = np.array([s[2] for s in spec_bins])
present_flux = np.array([s[3] for s in spec_bins])

# Ages to plot (matching Fig. 20 concept).
ages_plot = [0.1, 0.3, 1.0, 2.0, 4.6]
age_colors = ["purple", "blue", "green", "orange", "red"]
age_labels = ["0.1 Gyr (ZAMS)", "0.3 Gyr", "1.0 Gyr",
              "2.0 Gyr", "4.6 Gyr (Present Sun / 현재 태양)"]

fig, ax = plt.subplots(figsize=(11, 6))

for age, color, label in zip(ages_plot, age_colors, age_labels):
    # Enhancement factor at each wavelength for this age.
    enhancement = (age / 4.6) ** betas
    flux = present_flux * enhancement
    ax.semilogy(wavelengths, flux, "o-", color=color, lw=2, ms=5, label=label)

ax.set_xlabel("Wavelength [A] / 파장 [A]")
ax.set_ylabel("Relative Spectral Irradiance [arb.]\n상대 분광 복사 [임의 단위]")
ax.set_title("Solar Spectral Irradiance at Different Ages (Fig. 20 style)\n"
             "나이별 태양 분광 복사 (스펙트럼 경화/연화)")
ax.legend(fontsize=9, loc="lower right")
ax.set_xlim(1, 9000)
ax.set_xscale("log")
ax.set_ylim(0.005, 5e4)

# Annotate spectral regions.
regions = [
    (10, "X-ray"), (100, "EUV"), (1000, "FUV"), (2000, "UV"), (5000, "Vis")
]
for wl, name in regions:
    ax.axvline(wl, color="gray", ls=":", lw=0.5, alpha=0.5)
    ax.text(wl, ax.get_ylim()[1] * 0.7, name, fontsize=7, ha="center",
            color="gray")

# Arrow showing "spectral softening" direction.
ax.annotate("Spectral softening\n스펙트럼 연화",
            xy=(30, 20), xytext=(200, 5000),
            arrowprops=dict(arrowstyle="->", color="gray", lw=1.5),
            fontsize=9, color="gray")

plt.tight_layout()
plt.show()

# Print enhancement table (Table 6 style).
print("\nTable 6 Style — Enhancement Factors by Band and Age")
print("(Relative to present Sun / 현재 태양 대비)")
header = f"{'Band':<20}"
for age in ages_plot:
    header += f" {age:>7.1f} Gyr"
print(header)
print("-" * (20 + 11 * len(ages_plot)))
for wl, name, beta, _ in spec_bins[::2]:  # Show every other bin.
    row = f"{name + f' ({wl}A)':<20}"
    for age in ages_plot:
        enh = (age / 4.6) ** beta
        row += f" {enh:>10.1f}x"
    print(row)

## Part 5: Coronal Temperature Evolution
## 파트 5: 코로나 온도 진화

Güdel Eq. 15 shows a remarkably steep relation between average coronal temperature and X-ray luminosity:
Güdel Eq. 15는 평균 코로나 온도와 X선 광도 사이의 놀라울 정도로 가파른 관계를 보여줍니다:

$$L_X = 1.61 \times 10^{26}\,T_\text{av}^{4.05 \pm 0.25} \quad \text{[erg s}^{-1}\text{]}$$

This means a modest change in coronal temperature produces a huge change in X-ray luminosity. Physically, this reflects that more active (younger) stars don't just have "hotter coronae" — they develop **additional hot coronal structures** (8–20 MK) that boost X-ray emission.

이는 코로나 온도의 작은 변화가 X선 광도의 거대한 변화를 만든다는 것을 의미합니다. 물리적으로, 더 활동적인(젊은) 별이 단순히 "더 뜨거운 코로나"를 가지는 것이 아니라, X선 방출을 높이는 **추가적인 고온 코로나 구조** (8–20 MK)를 발전시킨다는 것을 반영합니다.

In [ ]:
# === Coronal Temperature vs X-ray Luminosity (Eq. 15) ===

# Eq. 15: L_X = 1.61e26 * T_av^{4.05}
T_av = np.logspace(np.log10(1e6), np.log10(30e6), 200)  # K
L_X_eq15 = 1.61e26 * (T_av / 1e6) ** 4.05  # T_av in MK for readability

# Actually Eq. 15 uses T_av in MK: L_X = 1.61e26 * T_av_MK^{4.05}.
# Let's recalculate properly: T_av in [K], so if paper uses [MK]:
# L_X = 1.61e26 * (T_av / 1e6)^4.05
T_av_MK = np.logspace(np.log10(1), np.log10(30), 200)
L_X_eq15 = 1.61e26 * T_av_MK ** 4.05

# "Sun in Time" sample approximate coronal temperatures (from EMD analyses).
sit_T_MK = np.array([10.0, 6.0, 5.5, 4.0, 3.0, 1.5, 1.2])  # Approx. MK
sit_LX_T = 1.61e26 * sit_T_MK ** 4.05

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# --- Left: L_X vs T_av ---
ax = axes[0]
ax.loglog(T_av_MK, L_X_eq15, "b-", lw=2.5,
          label=r"Eq. 15: $L_X = 1.61\times10^{26}\,T_\mathrm{av}^{4.05}$")
ax.scatter(sit_T_MK, sit_LX_T, c="red", s=80, zorder=5, edgecolors="k")
for name, t, lx in zip(sit_names, sit_T_MK, sit_LX_T):
    ax.annotate(name, (t, lx), textcoords="offset points",
                xytext=(6, 6), fontsize=8)
ax.set_xlabel(r"$T_\mathrm{av}$ [MK] / 평균 코로나 온도")
ax.set_ylabel(r"$L_X$ [erg s$^{-1}$]")
ax.set_title(r"Coronal Temperature vs $L_X$ (Eq. 15)" "\n"
             "코로나 온도 vs X선 광도")
ax.legend(fontsize=9)
ax.set_xlim(0.8, 40)
ax.set_ylim(1e26, 5e31)

# Shade saturation zone.
ax.axhline(L_X_sat, color="gray", ls="--", lw=1, alpha=0.7)
ax.text(1, L_X_sat * 1.5, "Saturation / 포화", fontsize=8, color="gray")

# --- Right: T_av vs age (combining Eq. 12 and Eq. 15) ---
# From L_X = 3e28 * t^{-1.5} and L_X = 1.61e26 * T^{4.05}:
# T^{4.05} = (3e28 / 1.61e26) * t^{-1.5}
# T = (3e28 / 1.61e26)^{1/4.05} * t^{-1.5/4.05}
age_grid = np.logspace(np.log10(0.1), np.log10(10), 200)
coeff = (3e28 / 1.61e26) ** (1.0 / 4.05)
T_from_age = coeff * age_grid ** (-1.5 / 4.05)  # in MK

ax = axes[1]
ax.loglog(age_grid, T_from_age, "r-", lw=2.5,
          label=r"$T_\mathrm{av} \propto t^{-0.37}$ (from Eqs. 12+15)")
ax.scatter(sit_ages_gyr, sit_T_MK, c="red", s=80, zorder=5, edgecolors="k")
for name, x, y in zip(sit_names, sit_ages_gyr, sit_T_MK):
    ax.annotate(name, (x, y), textcoords="offset points",
                xytext=(6, 6), fontsize=8)
ax.set_xlabel("Age [Gyr] / 나이 [Gyr]")
ax.set_ylabel(r"$T_\mathrm{av}$ [MK] / 평균 코로나 온도")
ax.set_title("Average Coronal Temperature vs Age\n평균 코로나 온도 vs 나이")
ax.legend(fontsize=9)
ax.set_xlim(0.08, 10)
ax.set_ylim(0.5, 30)

plt.tight_layout()
plt.show()

# Print the T-age relation.
print("\nCoronal Temperature Evolution")
print(f"{'Age [Gyr]':>10} {'T_av [MK]':>10} {'L_X [erg/s]':>14}")
print("-" * 38)
for t in [0.1, 0.3, 0.5, 1.0, 2.0, 4.6, 6.7]:
    T = coeff * t ** (-1.5 / 4.05)
    lx = 1.61e26 * T ** 4.05
    print(f"{t:>10.1f} {T:>10.1f} {lx:>14.2e}")

## Part 6: Faint Young Sun Paradox
## 파트 6: 젊고 어두운 태양 역설

Standard solar evolution models predict the ZAMS Sun was ~30% fainter ($L \approx 0.7\,L_\odot$) because the core's mean molecular weight was lower (less He accumulated). This would make early Earth ~20 K colder — frozen. Yet geologic evidence shows liquid water 3.8+ Gyr ago.

표준 태양 진화 모델에 따르면 ZAMS 태양은 ~30% 어두웠습니다 ($L \approx 0.7\,L_\odot$). 핵에 He이 덜 축적되어 평균 분자량이 낮았기 때문입니다. 이는 초기 지구를 ~20 K 더 춥게 만들어 완전히 얼어 있어야 했습니다. 그러나 지질학적 증거는 38억 년 전에 액체 물이 존재했음을 보여줍니다.

We plot: (1) $L_\text{bol}/L_{\odot}$ vs age (Fig. 37 style), (2) effective temperature, and (3) Earth surface temperature with different greenhouse scenarios. Also Fig. 38 style: solar flux with mass loss scenarios.

그래프: (1) 볼로메트릭 광도 vs 나이 (Fig. 37 스타일), (2) 유효 온도, (3) 다양한 온실 효과 시나리오에서의 지구 표면 온도. 또한 Fig. 38 스타일: 질량 손실 시나리오에서의 태양 플럭스.

In [ ]:
# === Faint Young Sun Paradox (Fig. 37 & 38 style) ===

# Standard solar evolution: approximate L/L_sun as linear increase.
# L(t) / L_sun ~ 1 / (1 + 2/5 * (1 - t/t_sun))  (Gough 1981)
# where t_sun = 4.6 Gyr and t is age.
t_sun = 4.6  # Gyr
age = np.linspace(0.01, 10, 500)

# Gough (1981) formula: L(t) = L_now / (1 + 2/5 * (1 - t/t_sun)).
L_ratio_standard = 1.0 / (1.0 + 0.4 * (1.0 - age / t_sun))

# Earth equilibrium temperature: T_eq = [(S(1-A))/(4*sigma)]^{1/4}.
# S = S_0 * L_ratio, S_0 = 1361 W/m^2 (solar constant).
S_0 = 1361.0  # W/m^2
A_earth = 0.30  # albedo
sigma_sb = 5.67e-8  # Stefan-Boltzmann

S_flux = S_0 * L_ratio_standard
T_eq = ((S_flux * (1 - A_earth)) / (4 * sigma_sb)) ** 0.25

# Greenhouse effect: present T_surface ~ 288 K, T_eq ~ 255 K.
# Delta_T_greenhouse ~ 33 K today.
T_eq_present = ((S_0 * (1 - A_earth)) / (4 * sigma_sb)) ** 0.25
dT_greenhouse_now = 288 - T_eq_present  # ~33 K

# Scenarios for surface temperature:
# 1) Constant greenhouse (33 K) — naive.
T_surf_const_gh = T_eq + dT_greenhouse_now
# 2) Enhanced CO2 greenhouse (~50 K): early atmosphere had more CO2.
T_surf_enhanced = T_eq + 50
# 3) Very strong CO2+CH4 greenhouse (~70 K).
T_surf_strong = T_eq + 70

# Mass loss scenarios (Fig. 38): if young Sun was more massive.
# M(t) = M_sun * (1 + delta_M * exp(-t/tau)) approximately.
# Higher mass -> higher luminosity (L ~ M^4 on MS).
def L_with_mass_loss(age_arr, delta_m, tau):
    """Compute L/L_sun with initial extra mass delta_m (in M_sun).

    Args:
        age_arr: Age array in Gyr.
        delta_m: Fractional extra mass at ZAMS (e.g. 0.03 for 3%).
        tau: e-folding timescale for mass loss in Gyr.

    Returns:
        L/L_sun ratio array.
    """
    # M(t) = 1 + delta_m * exp(-t/tau) for t < t_sun.
    m_ratio = 1.0 + delta_m * np.exp(-age_arr / tau)
    # L ~ M^4 approximately, combined with standard evolution.
    l_standard = 1.0 / (1.0 + 0.4 * (1.0 - age_arr / t_sun))
    return l_standard * m_ratio ** 4

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Left: L/L_sun vs age (Fig. 37 style) ---
ax = axes[0]
ax.plot(age, L_ratio_standard, "b-", lw=2.5,
        label="Standard (constant mass)")
ax.plot(age, L_with_mass_loss(age, 0.02, 1.0), "r--", lw=2,
        label=r"$\Delta M = 2\%$, $\tau=1$ Gyr")
ax.plot(age, L_with_mass_loss(age, 0.05, 0.5), "m--", lw=2,
        label=r"$\Delta M = 5\%$, $\tau=0.5$ Gyr")
ax.axhline(1.0, color="gray", ls=":", lw=0.8)
ax.axvline(t_sun, color="gray", ls=":", lw=0.8)
ax.scatter([t_sun], [1.0], marker="*", s=200, c="gold", edgecolors="k",
           zorder=5)
ax.set_xlabel("Age [Gyr] / 나이 [Gyr]")
ax.set_ylabel(r"$L / L_\odot$")
ax.set_title("Bolometric Luminosity vs Age (Fig. 37)\n볼로메트릭 광도 vs 나이")
ax.legend(fontsize=8)
ax.set_xlim(0, 8)
ax.set_ylim(0.6, 1.3)

# --- Middle: Solar flux at Earth vs age (Fig. 38 style) ---
ax = axes[1]
ax.plot(age, S_flux, "b-", lw=2.5, label="Standard model")
S_ml_2 = S_0 * L_with_mass_loss(age, 0.02, 1.0)
S_ml_5 = S_0 * L_with_mass_loss(age, 0.05, 0.5)
ax.plot(age, S_ml_2, "r--", lw=2, label=r"$\Delta M = 2\%$")
ax.plot(age, S_ml_5, "m--", lw=2, label=r"$\Delta M = 5\%$")
ax.axhline(S_0, color="gray", ls=":", lw=0.8)
ax.scatter([t_sun], [S_0], marker="*", s=200, c="gold", edgecolors="k",
           zorder=5)
ax.set_xlabel("Age [Gyr] / 나이 [Gyr]")
ax.set_ylabel("Solar Flux at 1 AU [W/m$^2$]\n1 AU에서의 태양 플럭스")
ax.set_title("Solar Flux at Earth (Fig. 38 style)\n지구에서의 태양 플럭스")
ax.legend(fontsize=8)
ax.set_xlim(0, 8)

# --- Right: Earth surface temperature scenarios ---
ax = axes[2]
ax.plot(age, T_eq, "b:", lw=1.5, label="No greenhouse / 온실 없음")
ax.plot(age, T_surf_const_gh, "b-", lw=2,
        label=r"Constant greenhouse ($\Delta T=33$ K)")
ax.plot(age, T_surf_enhanced, "orange", lw=2, ls="--",
        label=r"Enhanced CO$_2$ ($\Delta T=50$ K)")
ax.plot(age, T_surf_strong, "r-", lw=2,
        label=r"Strong CO$_2$+CH$_4$ ($\Delta T=70$ K)")
ax.axhline(273.15, color="cyan", ls="--", lw=1.5, alpha=0.7,
           label="Freezing point / 빙점")
ax.axhline(288, color="gray", ls=":", lw=0.8)
ax.fill_between(age, 250, 273.15, alpha=0.1, color="cyan")
ax.scatter([t_sun], [288], marker="*", s=200, c="gold", edgecolors="k",
           zorder=5)
ax.set_xlabel("Age [Gyr] / 나이 [Gyr]")
ax.set_ylabel("Surface Temperature [K]\n표면 온도 [K]")
ax.set_title("Earth Surface Temperature Scenarios\n"
             "지구 표면 온도 시나리오")
ax.legend(fontsize=7, loc="lower right")
ax.set_xlim(0, 8)
ax.set_ylim(220, 330)

plt.tight_layout()
plt.show()

# Print FYSP summary.
print("\nFaint Young Sun Paradox — Key Numbers")
L_zams = L_ratio_standard[np.argmin(np.abs(age - 0.01))]
T_zams = T_eq[np.argmin(np.abs(age - 0.01))]
print(f"ZAMS luminosity:        {L_zams:.2f} L_sun")
print(f"ZAMS T_eq (Earth):      {T_zams:.0f} K")
print(f"ZAMS T_surface (33K GH): {T_zams + 33:.0f} K  (< 273 K = frozen!)")
print(f"Present T_eq:            {T_eq_present:.0f} K")
print(f"Present T_surface:       {T_eq_present + 33:.0f} K")
print(f"\nTo reach 273 K at ZAMS with no extra factors:")
print(f"  Need greenhouse ΔT ≥ {273 - T_zams:.0f} K (vs 33 K today)")

## Part 7: Planetary Atmospheric Escape
## 파트 7: 행성 대기 탈출

The thermal escape parameter $X$ (Eq. 30) determines whether a planet loses its atmosphere via slow Jeans escape ($X \gg 1$) or catastrophic hydrodynamic blow-off ($X \lesssim 1.5$):

열적 탈출 파라미터 $X$ (Eq. 30)는 행성이 느린 Jeans 탈출($X \gg 1$)로 대기를 잃는지, 아니면 재앙적 유체역학적 blow-off($X \lesssim 1.5$)로 잃는지 결정합니다:

$$X = \frac{GMm}{kT_\text{exo}\,r}$$

We compute $X$ for hydrogen (H) and nitrogen (N₂) on Venus, Earth, and Mars as a function of exospheric temperature $T_\text{exo}$, showing how the young Sun's enhanced XUV heating pushed planets into the blow-off regime.

수소(H)와 질소(N₂)에 대해 금성, 지구, 화성의 $X$를 외기권 온도 $T_\text{exo}$의 함수로 계산하여, 젊은 태양의 향상된 XUV 가열이 행성을 blow-off 체제로 밀어넣었음을 보여줍니다.

In [ ]:
# === Planetary Atmospheric Escape (Eq. 30-31) ===

# Constants.
G = 6.674e-11       # m^3 kg^-1 s^-2
k_B = 1.381e-23     # J/K
m_H = 1.673e-27     # kg (hydrogen atom mass)
m_N2 = 28 * m_H     # kg (N2 molecule mass)
m_O = 16 * m_H      # kg (atomic oxygen mass)

# Planetary parameters: (name, mass [kg], exobase radius [m],
#                        present T_exo [K], color).
planets = {
    "Venus": {
        "M": 4.867e24, "r_exo": 6.20e6, "T_exo_now": 275,
        "color": "orange", "marker": "s"
    },
    "Earth": {
        "M": 5.972e24, "r_exo": 6.57e6, "T_exo_now": 1000,
        "color": "blue", "marker": "o"
    },
    "Mars": {
        "M": 6.417e23, "r_exo": 3.59e6, "T_exo_now": 300,
        "color": "red", "marker": "^"
    },
}


def escape_param(M, m, T_exo, r):
    """Compute the thermal escape parameter X (Eq. 30).

    Args:
        M: Planet mass [kg].
        m: Atmospheric particle mass [kg].
        T_exo: Exospheric temperature [K].
        r: Exobase radius [m].

    Returns:
        Dimensionless escape parameter X.
    """
    return G * M * m / (k_B * T_exo * r)


T_exo_range = np.logspace(np.log10(200), np.log10(20000), 300)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Left: X for hydrogen ---
ax = axes[0]
ax.set_title("Thermal Escape Parameter $X$ for Hydrogen (H)\n"
             "수소(H) 열적 탈출 파라미터")
for pname, pdata in planets.items():
    X_H = escape_param(pdata["M"], m_H, T_exo_range, pdata["r_exo"])
    ax.semilogy(T_exo_range, X_H, color=pdata["color"], lw=2.5,
                label=pname)
    # Mark present T_exo.
    X_now = escape_param(pdata["M"], m_H, pdata["T_exo_now"], pdata["r_exo"])
    ax.scatter([pdata["T_exo_now"]], [X_now], marker=pdata["marker"],
               s=120, c=pdata["color"], edgecolors="k", zorder=5)
    ax.annotate(f"{pname}\n(now)", (pdata["T_exo_now"], X_now),
                textcoords="offset points", xytext=(8, 0), fontsize=8)

# Blow-off threshold.
ax.axhline(1.5, color="gray", ls="--", lw=2, alpha=0.7)
ax.fill_between(T_exo_range, 0.01, 1.5, alpha=0.1, color="red")
ax.text(300, 0.8, "Blow-off regime\n유체역학적 탈출 영역",
        fontsize=9, color="red", style="italic")
ax.text(300, 5, "Jeans regime\nJeans 탈출 영역",
        fontsize=9, color="blue", style="italic")
ax.set_xlabel(r"$T_\mathrm{exo}$ [K] / 외기권 온도")
ax.set_ylabel("$X$ (escape parameter / 탈출 파라미터)")
ax.legend(fontsize=9, loc="upper right")
ax.set_xlim(200, 20000)
ax.set_ylim(0.1, 100)
ax.set_xscale("log")

# Annotate young Sun T_exo range.
ax.axvspan(3000, 15000, alpha=0.08, color="purple")
ax.text(6000, 50, "Young Sun\nXUV heating\n젊은 태양\nXUV 가열",
        fontsize=8, ha="center", color="purple")

# --- Right: X for nitrogen (N2) ---
ax = axes[1]
ax.set_title("Thermal Escape Parameter $X$ for Nitrogen (N$_2$)\n"
             "질소(N₂) 열적 탈출 파라미터")
for pname, pdata in planets.items():
    X_N2 = escape_param(pdata["M"], m_N2, T_exo_range, pdata["r_exo"])
    ax.semilogy(T_exo_range, X_N2, color=pdata["color"], lw=2.5,
                label=pname)
    X_now = escape_param(pdata["M"], m_N2, pdata["T_exo_now"],
                         pdata["r_exo"])
    ax.scatter([pdata["T_exo_now"]], [X_now], marker=pdata["marker"],
               s=120, c=pdata["color"], edgecolors="k", zorder=5)
    ax.annotate(f"{pname}\n(now)", (pdata["T_exo_now"], X_now),
                textcoords="offset points", xytext=(8, 0), fontsize=8)

ax.axhline(1.5, color="gray", ls="--", lw=2, alpha=0.7)
ax.fill_between(T_exo_range, 0.01, 1.5, alpha=0.1, color="red")
ax.text(8000, 0.8, "Blow-off\n탈출", fontsize=9, color="red", style="italic")
ax.set_xlabel(r"$T_\mathrm{exo}$ [K] / 외기권 온도")
ax.set_ylabel("$X$ (escape parameter / 탈출 파라미터)")
ax.legend(fontsize=9, loc="upper right")
ax.set_xlim(200, 20000)
ax.set_ylim(0.1, 10000)
ax.set_xscale("log")

ax.axvspan(3000, 15000, alpha=0.08, color="purple")
ax.text(6000, 3000, "Young Sun\nXUV heating",
        fontsize=8, ha="center", color="purple")

plt.tight_layout()
plt.show()

# Summary table.
print("\nThermal Escape Parameter X at Present T_exo")
print("열적 탈출 파라미터 X (현재 외기권 온도에서)")
print(f"{'Planet':<10} {'T_exo [K]':>10} {'X(H)':>10} {'X(N2)':>10} "
      f"{'X(O)':>10}")
print("-" * 55)
for pname, pdata in planets.items():
    xh = escape_param(pdata["M"], m_H, pdata["T_exo_now"], pdata["r_exo"])
    xn = escape_param(pdata["M"], m_N2, pdata["T_exo_now"], pdata["r_exo"])
    xo = escape_param(pdata["M"], m_O, pdata["T_exo_now"], pdata["r_exo"])
    print(f"{pname:<10} {pdata['T_exo_now']:>10} {xh:>10.1f} {xn:>10.0f} "
          f"{xo:>10.0f}")

print("\n--- Interpretation / 해석 ---")
print("H on Mars: X is small → significant Jeans escape even today.")
print("H on Earth: X ~ 7 → slow Jeans escape (protected by magnetosphere).")
print("N2 on all planets: X >> 1 → safe from thermal escape today.")
print("But with young Sun (T_exo ~ 5000-10000 K):")
for pname, pdata in planets.items():
    for T_young in [5000, 10000]:
        xh = escape_param(pdata["M"], m_H, T_young, pdata["r_exo"])
        print(f"  {pname} H at {T_young}K: X = {xh:.2f}"
              f" {'→ BLOW-OFF!' if xh < 1.5 else ''}")